# Day 33 — Performance Optimization: pandas
**Estimated time:** 90 minutes  
**Learning objectives:**
1. Benchmark `.iterrows()` vs vectorisation and quantify the difference
2. Use category dtype and chunking to reduce memory footprint
3. Profile memory usage and understand pandas performance trade-offs

---
> **Context:** In senior analytics work, you're not just writing code — you're reviewing it. Understanding pandas performance lets you spot bottlenecks in junior analysts' code during code reviews and provide concrete guidance. The goal is not premature optimisation, but knowing when to care.


## 1. Benchmark: .iterrows() vs Vectorisation

In [ ]:
import pandas as pd
import numpy as np
import time
from pathlib import Path

from analytics_bootcamp.config import RAW_DATA_DIR as DATA
mat = pd.read_csv(DATA / 'materials_inventory.csv')

# Expand dataset for meaningful benchmarks
mat_large = pd.concat([mat] * 50, ignore_index=True)  # 20,000 rows
print(f'Benchmark dataset: {len(mat_large):,} rows')

# ── Method 1: iterrows (slow) ──────────────────────────────────
start = time.perf_counter()
values = []
for _, row in mat_large.iterrows():
    values.append(row['LABST'] * row['STPRS'])
mat_large['INV_VALUE_ITER'] = values
t_iter = time.perf_counter() - start

# ── Method 2: .apply (moderate) ───────────────────────────────
start = time.perf_counter()
mat_large['INV_VALUE_APPLY'] = mat_large.apply(lambda r: r['LABST'] * r['STPRS'], axis=1)
t_apply = time.perf_counter() - start

# ── Method 3: Vectorised (fast) ───────────────────────────────
start = time.perf_counter()
mat_large['INV_VALUE_VEC'] = mat_large['LABST'] * mat_large['STPRS']
t_vec = time.perf_counter() - start

print(f'iterrows:    {t_iter:.4f}s')
print(f'apply:       {t_apply:.4f}s')
print(f'vectorised:  {t_vec:.4f}s')
print(f'Speedup (iter vs vec): {t_iter/t_vec:.1f}x')
print(f'Speedup (apply vs vec): {t_apply/t_vec:.1f}x')

# Verify all methods agree
assert (mat_large['INV_VALUE_ITER'] - mat_large['INV_VALUE_VEC']).abs().max() < 0.01
print('\nAll methods produce identical results.')

## 2. Category Dtype — Cardinality Savings

In [ ]:
# YOUR SOLUTION
import sys

so = pd.read_csv(DATA / 'sales_orders.csv')
hr = pd.read_csv(DATA / 'hr_headcount.csv')

# Before: object dtype for string columns
mem_before = so.memory_usage(deep=True).sum()

# After: convert low-cardinality string columns to category
low_card_cols = ['STATUS', 'WAERK', 'VKORG', 'VTWEG', 'SPART']
so_cat = so.copy()
for col in low_card_cols:
    so_cat[col] = so_cat[col].astype('category')

mem_after = so_cat.memory_usage(deep=True).sum()

print(f'Before (object):   {mem_before/1024:,.1f} KB')
print(f'After (category):  {mem_after/1024:,.1f} KB')
print(f'Savings:           {(mem_before-mem_after)/1024:,.1f} KB ({(1-mem_after/mem_before)*100:.1f}%)')
print()

# Show cardinality for each column
for col in low_card_cols:
    print(f'  {col}: {so[col].nunique()} unique values → category dtype saves ~{(so[col].memory_usage(deep=True) - so_cat[col].memory_usage(deep=True)) / 1024:.1f} KB')

## 3. Chunking Large CSVs

In [ ]:
# YOUR SOLUTION
# Simulate processing a large file in chunks (important when file doesn't fit in RAM)
# Real scenario: 10M row SAP extract that's 2GB — must be chunked

import io

# Create a 'large' version of sales orders
large_csv_path = '/tmp/large_sales.csv'
pd.concat([pd.read_csv(DATA / 'sales_orders.csv')] * 200).to_csv(large_csv_path, index=False)
print(f'Large file size: {Path(large_csv_path).stat().st_size / 1024 / 1024:.1f} MB')

CHUNK_SIZE = 5000
chunk_results = []

for chunk in pd.read_csv(large_csv_path, chunksize=CHUNK_SIZE):
    # Only keep what we need — aggregate per chunk
    open_chunk = chunk[chunk['STATUS'] == 'OPEN']
    chunk_agg = open_chunk.groupby('VKORG')['NETWR'].sum().reset_index()
    chunk_results.append(chunk_agg)

# Combine chunk results
final = (
    pd.concat(chunk_results)
    .groupby('VKORG')['NETWR'].sum()
    .reset_index()
    .sort_values('NETWR', ascending=False)
)

print('\nOpen order backlog by sales org (from chunked processing):')
print(final.to_string(index=False))

## 4. .query() Performance

In [ ]:
# YOUR SOLUTION
mat_large = pd.read_csv(DATA / 'materials_inventory.csv')
mat_large = pd.concat([mat_large] * 50, ignore_index=True)

# Method 1: Boolean indexing
start = time.perf_counter()
for _ in range(20):
    _ = mat_large[(mat_large['WERKS'] == 1000) & (mat_large['LABST'] > 100)]
t_bool = (time.perf_counter() - start) / 20

# Method 2: .query()
start = time.perf_counter()
for _ in range(20):
    _ = mat_large.query('WERKS == 1000 and LABST > 100')
t_query = (time.perf_counter() - start) / 20

print(f'Boolean indexing avg: {t_bool*1000:.2f} ms')
print(f'.query() avg:         {t_query*1000:.2f} ms')

# Note: .query() advantage grows with more complex expressions and with numexpr installed
# Main benefit: readability for complex multi-condition filters

## 5. Memory Profiling with sys.getsizeof

In [ ]:
# YOUR SOLUTION
import sys

def df_memory_report(df: pd.DataFrame, name: str = 'DataFrame') -> pd.DataFrame:
    """Return a per-column memory usage report."""
    mem = df.memory_usage(deep=True)
    report = pd.DataFrame({
        'column': mem.index,
        'dtype': ['index'] + [str(df[c].dtype) for c in df.columns],
        'memory_kb': (mem / 1024).round(2),
        'unique_values': [df.index.nunique()] + [df[c].nunique() for c in df.columns]
    })
    total_kb = mem.sum() / 1024
    print(f'\n=== Memory Report: {name} (Total: {total_kb:.1f} KB) ===')
    return report.sort_values('memory_kb', ascending=False)

# Compare memory before/after optimizations
so_raw = pd.read_csv(DATA / 'sales_orders.csv')
so_opt = so_raw.copy()
so_opt['STATUS'] = so_opt['STATUS'].astype('category')
so_opt['WAERK'] = so_opt['WAERK'].astype('category')
so_opt['VKORG'] = so_opt['VKORG'].astype('category')
so_opt['NETWR'] = so_opt['NETWR'].astype('float32')   # halves memory vs float64
so_opt['MENGE'] = so_opt['MENGE'].astype('int32')

report_raw = df_memory_report(so_raw, 'sales_orders RAW')
report_opt = df_memory_report(so_opt, 'sales_orders OPTIMIZED')

print(report_raw.to_string(index=False))
print()
print(report_opt.to_string(index=False))

## Daily Prompt
**Answer independently:**

1. Create a `optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame` function that automatically:
   - Converts string columns with <50 unique values to `category`
   - Downcasts integer columns to the smallest fitting integer type using `pd.to_numeric(..., downcast='integer')`
   - Returns the optimised DataFrame and prints the memory savings
2. Benchmark `.groupby().agg()` vs a manual Python loop for computing the sum of `NETWR` per `VKORG` across 100,000 rows. How many times faster is pandas?
3. When should you NOT use chunking? Give two concrete examples where chunking would either fail or produce incorrect results.
